# Monte Carlo with GBM

`VanillaMCPricer` with `GeometricBrownianMotion`: the model built from the market, convergence and standard error,
stateless seeds, variance reduction, and greeks by AAD (pathwise) against Black–Scholes and bump-and-reprice.
Docs: *Models and Pricing* (Monte Carlo framework), *Architecture* (Models), *Exercises* (`equity.tex`).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq

## Market, option, reference
TICKER1, 1Y call at the forward. Reference: `BlackScholesPricer(dividend_model="discrete")`, the same model as the GBM
built by `GeometricBrownianMotion.from_equity` (escrowed spot $S - \mathrm{PV}$, carry = repo, $r = -\ln P(0,T)/T$, vol at $(K,T)$).

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
EXPIRY = EVALUATION_DATE + timedelta(days=365)
K = round(float(eq.forward(EXPIRY)), 2)


def call():
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=EXPIRY, option_type=tq.OptionType.Call,
                            strike=K, underlying=eq.ticker)


ref = call()
bs = tq.BlackScholesPricer(dividend_model="discrete")
bs.price(ref, mkt, autodiff=True)
BS = float(ref.price)
model = tq.GeometricBrownianMotion.from_equity(eq, EXPIRY, K)
print(f"Black-Scholes = {BS:.4f}")
T = eq.year_fraction(EXPIRY)
print(f"GBM from the market: x0 = S - PV(div) = {float(model.initial_values()):.4f}, "
      f"vol at (K, T) = {float(eq.vol_surface.volatility(K, T)):.4%}")

## Convergence: error $\propto 1/\sqrt N$
The estimator is the mean of $N$ discounted payoffs: its standard error is $\mathrm{std}/\sqrt N$ (`product.standard_error`).
GBM is simulated exactly, so one time step is enough for a European payoff.

In [ ]:
def mc(n_paths, n_steps=1, seed=42, model=tq.GeometricBrownianMotion, autodiff=False):
    o = call()
    pricer = tq.VanillaMCPricer(model=model, n_paths=n_paths, n_steps=n_steps, seed=seed)
    pricer.price(o, mkt, autodiff=autodiff)
    return o, pricer


rows = []
for n in (1_000, 4_000, 16_000, 64_000, 256_000):
    o, _ = mc(n)
    rows.append({"paths": n, "price": float(o.price), "error vs BS": float(o.price) - BS,
                 "std error": float(o.standard_error), "error / s.e.": (float(o.price) - BS) / float(o.standard_error)})
conv = pd.DataFrame(rows).set_index("paths")
conv.round(4)

In [ ]:
seeds = range(20)
ns = np.array([1_000, 4_000, 16_000, 64_000])
spread = [np.std([float(mc(n, seed=s)[0].price) for s in seeds]) for n in ns]
plt.figure(figsize=(6, 4))
plt.loglog(ns, spread, "o-", label="std of the price over 20 seeds")
plt.loglog(ns, spread[0] * np.sqrt(ns[0] / ns), "--", label=r"$\propto 1/\sqrt{N}$")
plt.xlabel("paths"); plt.grid(alpha=0.3, which="both"); plt.legend();

## Seeds and time steps
Draws are stateless (`tf.random.stateless_normal` with the seed): same seed, same paths, same price; a bump repriced
with the same seed uses **common random numbers**. The number of steps does not change the GBM law (exact
simulation): 252 steps cost 252 times the normals of 1.

In [ ]:
import time
a, b, c = mc(50_000, seed=7)[0], mc(50_000, seed=7)[0], mc(50_000, seed=8)[0]
print(f"seed 7 twice: {float(a.price):.6f} {float(b.price):.6f};  seed 8: {float(c.price):.6f}")
for steps in (1, 12, 252):
    t0 = time.time(); o, _ = mc(100_000, n_steps=steps); dt = time.time() - t0
    print(f"{steps:4d} steps: price {float(o.price):.4f} +/- {float(o.standard_error):.4f}  ({dt:.2f}s)")

## Variance reduction
Not in the pricer yet; shown here on the model directly (`model.evolve(t_grid, dw)`):

- **antithetic variates**: use $Z$ and $-Z$ and average the two payoffs (negatively correlated);
- **control variate**: $S_T$ has a known mean, the forward: $\hat C = \overline{C} - \beta\,(\overline{S_T} - F)$,
  $\beta = \mathrm{cov}(C, S_T)/\mathrm{var}(S_T)$.

In [ ]:
n = 50_000
DF, F = float(eq.discount_curve.discount(EXPIRY)), float(eq.forward(EXPIRY))
z = tf.random.stateless_normal([n, 1], seed=[1, 0], dtype=tf.float64)
grid = tf.constant([T], tf.float64)
s_plain = model.evolve(grid, tf.random.stateless_normal([2 * n, 1], seed=[1, 0], dtype=tf.float64))[:, 0].numpy()
s_up, s_dn = model.evolve(grid, z)[:, 0].numpy(), model.evolve(grid, -z)[:, 0].numpy()
pay = lambda s: DF * np.maximum(s - K, 0.0)

plain = pay(s_plain)
anti = 0.5 * (pay(s_up) + pay(s_dn))
beta = np.cov(plain, s_plain)[0, 1] / np.var(s_plain)
control = plain - beta * (s_plain - F)                                 # E[S_T] = F under the T-forward measure
rows = [{"method": "plain (2N paths)", "price": plain.mean(), "s.e.": plain.std() / np.sqrt(2 * n)},
        {"method": "antithetic (N pairs)", "price": anti.mean(), "s.e.": anti.std() / np.sqrt(n)},
        {"method": "control variate (2N)", "price": control.mean(), "s.e.": control.std() / np.sqrt(2 * n)}]
print(f"Black-Scholes {BS:.4f}")
pd.DataFrame(rows).set_index("method").round(4)

## Greeks by AAD
One backward pass on the tape gives the pathwise derivative of the mean payoff w.r.t. every market variable the
model was built from: delta (spot), vega per node (surface), rho per pillar (curve), dividend amounts. Pathwise works
because the call payoff is Lipschitz (it would fail for a digital: see `digital_options`).

In [ ]:
o, pricer = mc(200_000, autodiff=True)
g = pricer.tape.gradient(o.price, eq.risk_factors())
g_bs = bs.tape.gradient(ref.price, eq.risk_factors())
key = lambda t: f"EQ:EUR:TICKER1:{t}"
rows = [
    {"greek": "delta", "MC (AAD)": float(g[key("SPOT")][0]), "Black-Scholes": float(g_bs[key("SPOT")][0])},
    {"greek": "vega (sum of nodes)", "MC (AAD)": float(tf.reduce_sum(g[key("VOL")][0])),
     "Black-Scholes": float(tf.reduce_sum(g_bs[key("VOL")][0]))},
    {"greek": "rho (sum of pillars)", "MC (AAD)": sum(float(x) for x in g[eq.discount_curve.name] if x is not None),
     "Black-Scholes": sum(float(x) for x in g_bs[eq.discount_curve.name] if x is not None)},
    {"greek": "dV/dD (first dividend)", "MC (AAD)": float(g[key("DIV")][0][0]), "Black-Scholes": float(g_bs[key("DIV")][0][0])},
]
pd.DataFrame(rows).set_index("greek").round(4)

## AAD vs bump-and-reprice
Finite differences need one repricing per input (2 for central): with **independent** seeds the noise of the two
prices does not cancel and the estimate blows up as the bump shrinks; with **common random numbers** it is as
accurate as the pathwise one. AAD gets all the greeks for the cost of a few prices.

In [ ]:
spot, s0 = eq.spot, float(eq.spot)


def bumped_delta(h, crn):
    out = []
    for sgn, seed in ((1, 11), (-1, 11 if crn else 12)):
        spot.assign(s0 + sgn * h)
        out.append(float(mc(100_000, seed=seed)[0].price))
    spot.assign(s0)
    return (out[0] - out[1]) / (2 * h)


o, p = mc(100_000, seed=11, autodiff=True)
aad = float(p.tape.gradient(o.price, spot))
rows = [{"bump": h, "CRN": bumped_delta(h, True), "independent seeds": bumped_delta(h, False)} for h in (5.0, 1.0, 0.1)]
print(f"AAD delta (seed 11): {aad:.5f}   Black-Scholes: {float(g_bs[key('SPOT')][0]):.5f}")
pd.DataFrame(rows).set_index("bump").round(5)

## Exercises
1. How many paths are needed for a standard error of 1 cent on this call? And with the control variate?
2. Price a deep out-of-the-money call (K = 1.5 F): what happens to the relative standard error, and why?
3. Compute the AAD gamma of the Monte Carlo price (second tape): is it reliable? Why is gamma harder than delta for
   pathwise derivatives?